In [ ]:
%pip install dill ipython-autotime flaml[automl] numpy==1.24.4

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%load_ext autotime
import pathlib, google, logging, dill, dataclasses, numpy as np, pandas as pd, sklearn as sk, matplotlib.pyplot as plt, seaborn as sns
from copy import deepcopy as copy
from sklearn.impute import SimpleImputer #https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import make_scorer, f1_score, classification_report, ConfusionMatrixDisplay, RocCurveDisplay
from sklearn.inspection import permutation_importance
from flaml import AutoML

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (f1_score, make_scorer, accuracy_score, precision_score,
                             recall_score, confusion_matrix, classification_report,
                             roc_curve, auc, roc_auc_score, balanced_accuracy_score)
from sklearn.inspection import permutation_importance
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from flaml import AutoML
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path



sk.set_config(transform_output="pandas") #https://scikit-learn.org/stable/auto_examples/miscellaneous/plot_set_output.html
pd.set_option('display.max_columns', None)
logging.getLogger("flaml.tune.searcher.blendsearch").setLevel(logging.WARNING)
google.colab.drive.mount('/content/drive')
root = pathlib.Path('/content/drive/MyDrive/Data science II')
seed = 42

def disp(X, max_rows=3, precision=None, **props):
    """convenient display method"""
    props = {
        'text-align': 'center',
        'vertical-align': 'top',
        'border': '1px solid white',
        'width': 'auto',
        } | props
    fmt = {'precision': precision, 'hyperlinks': 'html'}
    try:
        display(
            pd.DataFrame(X)
            .head(max_rows)
            .style
            .format(**fmt)
            .format_index(**fmt, axis=0)
            .format_index(**fmt, axis=1)
            .highlight_null()
            .set_table_styles([{'selector':k, 'props':[*props.items()]} for k in ['th','td']])
        )
    except:
        print(X)

def rm(path, root=False):
    path = pathlib.Path(path)
    if path.is_file():
        path.unlink()
    elif path.is_dir():
        if root:
            shutil.rmtree(path)
        else:
            for p in path.iterdir():
                rm(p, True)

def mkdir(path):
    path = pathlib.Path(path)
    folder = path if path.suffix == '' else path.parent
    folder.mkdir(parents=True, exist_ok=True)
    return path

def reset(path):
    rm(path)
    return mkdir(path)

def dump(path, obj, **kwargs):
    path = reset(path)
    print(f'dump to {path}')
    if not isinstance(obj, pd.DataFrame):
        with open(path, 'wb') as f:
            return dill.dump(obj, f, **kwargs)
    elif path.suffix == '.parquet':
        obj.to_parquet(path, **kwargs)
    elif path.suffix == '.csv':
        obj.to_csv(path, **kwargs)
    else:
        raise Exception('path suffix must be .parquet or .csv if obj is a DataFrame')

def load(path, **kwargs):
    path = pathlib.Path(path)
    if path.suffix == '.parquet':
        return pd.read_parquet(path, **kwargs)
    elif path.suffix == '.csv':
        return pd.read_csv(path, **kwargs)
    else:
        with open(path, 'rb') as f:
            return dill.load(f, **kwargs)



def subgroup_analysis(model, X, y, categorical_features):
    """Analyze model performance across categorical feature subgroups"""
    results = []
    for feature in categorical_features:
        groups = X[feature].unique()
        for group in groups:
            mask = X[feature] == group
            X_sub = X[mask]
            y_sub = y[mask]

            # Skip if subgroup is too small or has only one true class
            if len(y_sub) < 10 or y_sub.nunique() < 2:
                continue

            pred = model.predict(X_sub)

            # Calculate metrics with labels parameter to ensure both classes are considered
            report = classification_report(
                y_sub, pred,
                output_dict=True,
                zero_division=0,
                labels=[0, 1]  # Force both classes
            )

            result = {
                'feature': feature,
                'group': group,
                'n_samples': len(y_sub),
                'accuracy': report['accuracy'],
                'class_0_count': int((y_sub == 0).sum()),
                'class_1_count': int((y_sub == 1).sum()),
                'pred_0_count': int((pred == 0).sum()),
                'pred_1_count': int((pred == 1).sum()),
            }

            # Add metrics for both classes
            for label in [0, 1]:
                label_str = str(label)
                if label_str in report:
                    result.update({
                        f'precision_{label}': report[label_str]['precision'],
                        f'recall_{label}': report[label_str]['recall'],
                        f'f1_{label}': report[label_str]['f1-score'],
                    })
                else:
                    result.update({
                        f'precision_{label}': 0.0,
                        f'recall_{label}': 0.0,
                        f'f1_{label}': 0.0,
                    })
            results.append(result)

    return pd.DataFrame(results)

In [ ]:
#Read datadiabetes_012_health_indicators_BRFSS2015.csv
df = pd.read_csv(root / '/content/drive/MyDrive/Data science II/MyDataDiabetes.csv')
cf = pd.read_csv(root / '/content/drive/MyDrive/Data science II/DP05Data1.csv')

# Relabel 'diabetes' column from 1 and 2 to 0 and 1
df['diabetes'] = df['diabetes'].replace({1: 1, 2: 0})
# ----------------------------- Exploratory Data Analysis -----------------------------

# Identify columns
continuous = ['AGE','AnnualHouseholdIncome','bmi']
categorical = ['YourGeneralHealth','InsuranceStatus', 'PhyActCate','TakingSubsc','HeartDisease','HadStroke','CardioDisease','AsthmaDisease','AnyCncer','SEX','HispanicLatinoOrigin','MarStatus','EduStatus','EmployStatus','DisabilityStatus','SmoleCigga','cfips','state']
target = 'diabetes'

# 1. Missing values
print("\nMissing Values:")
missing = df.isnull().sum()

print(missing[missing > 0])

# descriptives
print("\nDescriptive Statistics:")
print(df.describe())

# 2. Class distribution
print("\nClass Distribution:")
print(df[target].value_counts(normalize=True))
sns.countplot(data=df, x=target)
plt.title("Target Class Distribution")
plt.show()

# 3. Distribution of continuous features
df[continuous].hist(figsize=(16, 12), bins=20, layout=(4, 4))
plt.suptitle("Histograms of Continuous Features", fontsize=16)
plt.tight_layout()
plt.show()

# 4. Distribution of categorical features
for col in categorical:
    sns.countplot(data=df, x=col, hue=target)
    plt.title(f"{col} by {target}")
    plt.show()

# 5. Correlation matrix of continuous features
plt.figure(figsize=(12, 10))
corr_matrix = df[continuous].corr()
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='coolwarm')
plt.title("Correlation Matrix (Continuous Features)")
plt.show()

In [ ]:
sk.set_config(transform_output="pandas") #https://scikit-learn.org/stable/auto_examples/miscellaneous/plot_set_output.html
pd.set_option('display.max_columns', None)
logging.getLogger("flaml.tune.searcher.blendsearch").setLevel(logging.WARNING)
google.colab.drive.mount('/content/drive')
root = pathlib.Path('/content/drive/MyDrive/Data science II')
seed = 42

def disp(X, max_rows=3, precision=None, **props):
    """convenient display method"""
    props = (
        {
            'text-align': 'center',
            'vertical-align': 'top',
            'border': '1px solid white',
            'width': 'auto',
        } | props
    )
    fmt = {'precision': precision, 'hyperlinks': 'html'}
    try:
        display(
            pd.DataFrame(X)
            .head(max_rows)
            .style
            .format(**fmt)
            .format_index(**fmt, axis=0)
            .format_index(**fmt, axis=1)
            .highlight_null()
            .set_table_styles([{'selector':k, 'props':[*props.items()]} for k in ['th','td']])
        )
    except:
        print(X)

def rm(path, root=False):
    path = pathlib.Path(path)
    if path.is_file():
        path.unlink()
    elif path.is_dir():
        if root:
            shutil.rmtree(path)
        else:
            for p in path.iterdir():
                rm(p, True)

def mkdir(path):
    path = pathlib.Path(path)
    folder = path if path.suffix == '' else path.parent
    folder.mkdir(parents=True, exist_ok=True)
    return path

def reset(path):
    rm(path)
    return mkdir(path)

def dump(path, obj, **kwargs):
    path = reset(path)
    print(f'dump to {path}')
    if not isinstance(obj, pd.DataFrame):
        with open(path, 'wb') as f:
            return dill.dump(obj, f, **kwargs)
    elif path.suffix == '.parquet':
        obj.to_parquet(path, **kwargs)
    elif path.suffix == '.csv':
        obj.to_csv(path, **kwargs)
    else:
        raise Exception('path suffix must be .parquet or .csv if obj is a DataFrame')

def load(path, **kwargs):
    path = pathlib.Path(path)
    if path.suffix == '.parquet':
        return pd.read_parquet(path, **kwargs)
    elif path.suffix == '.csv':
        return pd.read_csv(path, **kwargs)
    else:
        with open(path, 'rb') as f:
            return dill.load(f, **kwargs)


def subgroup_analysis(model, X, y, categorical_features):
    """Analyze model performance across categorical feature subgroups"""
    results = []
    for feature in categorical_features:
        groups = X[feature].unique()
        for group in groups:
            mask = X[feature] == group
            X_sub = X[mask]
            y_sub = y[mask]

            # Skip if subgroup is too small or has only one true class
            if len(y_sub) < 10 or y_sub.nunique() < 2:
                continue

            pred = model.predict(X_sub)

            # Calculate metrics with labels parameter to ensure both classes are considered
            report = classification_report(
                y_sub, pred,
                output_dict=True,
                zero_division=0,
                labels=[0, 1]  # Force both classes
            )

            result = {
                'feature': feature,
                'group': group,
                'n_samples': len(y_sub),
                'accuracy': report['accuracy'],
                'class_0_count': int((y_sub == 0).sum()),
                'class_1_count': int((y_sub == 1).sum()),
                'pred_0_count': int((pred == 0).sum()),
                'pred_1_count': int((pred == 1).sum()),
            }

            # Add metrics for both classes
            for label in [0, 1]:
                label_str = str(label)
                if label_str in report:
                    result.update({
                        f'precision_{label}': report[label_str]['precision'],
                        f'recall_{label}': report[label_str]['recall'],
                        f'f1_{label}': report[label_str]['f1-score'],
                    })
                else:
                    result.update({
                        f'precision_{label}': 0.0,
                        f'recall_{label}': 0.0,
                        f'f1_{label}': 0.0,
                    })
            results.append(result)

    return pd.DataFrame(results)

@dataclasses.dataclass
class diab():
    wrangler: any
    learner: any
    param_grid: any
    scorer: any
    target: str = 'diabetes'
    n_splits: int = 5
    refit_time: int = 120
    n_repeats: int = 5
    path: str = None


    def __post_init__(self):
        self.now = pd.Timestamp.now()
        self.path = mkdir(self.path if self.path is not None else root/f'models/{self.now}.pkl')

        #make modeling & holdout sets
        X = copy(df)
        y = X.pop(self.target)
        # Drop rows with missing target values before splitting
        valid_indices = y.dropna().index
        X = X.loc[valid_indices]
        y = y.loc[valid_indices]

        self.X_modeling, self.X_holdout, self.y_modeling, self.y_holdout = train_test_split(X, y, test_size=0.20, stratify=y, random_state=seed)

        #make estimator & grid
        self.learner.__sklearn_tags__ = getattr(sk.base, self.learner._estimator_type.capitalize()+'Mixin')().__sklearn_tags__  # stupid hack to correct FLAML/sklearn bug in ConfusionMatrixDisplay
        self.estimator = Pipeline((('wrangle', self.wrangler), ('learn', self.learner)))
        self.grid = GridSearchCV(self.estimator, self.param_grid, cv=self.n_splits, scoring=self.scorer, refit=False)


    def train(self):
        #run grid search
        self.grid.fit(self.X_modeling, self.y_modeling)

        #retrain best estimator with larger time_budget and cross-validation if FLAML
        kwargs = {
            'time_budget': self.refit_time,
            'retrain_full': True,
            'eval_method': 'cv',
            'n_splits': self.n_splits,
        }
        kwargs = {f'learn__{k}':v for k,v in kwargs.items() if k in self.learner.get_params()}
        self.estimator = self.estimator.set_params(**self.grid.best_params_).fit(self.X_modeling, self.y_modeling, **kwargs)

        #predict
        self.pred_modeling = self.estimator.predict(self.X_modeling)
        self.pred_holdout  = self.estimator.predict(self.X_holdout)


    def evaluate(self):
        #name of best algorithm
        self.algorithm = getattr(self.estimator['learn'], 'best_estimator', str(self.estimator['learn']).split('(')[0])
        disp(self.algorithm)

        #all results from grid search
        self.grid.cv_df = pd.DataFrame(self.grid.cv_results_['params']).assign(cv_score=self.grid.cv_results_['mean_test_score']).sort_values('cv_score', ascending=False).rename_axis(index='grid_id')
        disp(self.grid.cv_df, None)

        #best estimator's scores
        self.scores = {'cv':self.grid.best_score_, 'holdout':self.scorer(self.estimator, self.X_holdout, self.y_holdout)}
        disp(self.scores)

        #reports
        self.classification_report = classification_report(self.y_holdout, self.pred_holdout)
        disp(self.classification_report)
        ConfusionMatrixDisplay.from_estimator(self.estimator, self.X_holdout, self.y_holdout)
        RocCurveDisplay.from_estimator(self.estimator, self.X_holdout, self.y_holdout)
        plt.show()

        #feature importance (permutation)
        imp = permutation_importance(self.estimator, self.X_modeling, self.y_modeling, n_repeats=self.n_repeats, random_state=seed)
        self.importances = pd.DataFrame(imp.importances.T, columns=self.X_modeling.columns)
        mu = self.importances.mean().sort_values(ascending=False)
        # sns.swarmplot(self.importances[mu[mu>0].index], orient='h')
        sns.stripplot(self.importances[mu[mu>0].index], orient='h')
        plt.xlabel('permutation importance')
        plt.show()


        # Subgroup analysis
        print("\nSubgroup Performance Analysis:")
        subgroup_df = subgroup_analysis(
            self.estimator, self.X_holdout, self.y_holdout, categorical)
        display(subgroup_df.style.background_gradient(
            cmap='Blues', subset=['accuracy', 'f1_0', 'f1_1']))

        plt.figure(figsize=(12, 6))
        sns.barplot(data=subgroup_df, x='group', y='f1_1', hue='feature')
        plt.title('F1 Score for DR-Positive Class Across Subgroups')
        plt.ylabel('F1 Score')
        plt.xlabel('Subgroup')
        plt.xticks(rotation=45)
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.tight_layout()
        plt.show()

    def run(self):
        print('\n' + '#'*80)
        self.train()
        self.evaluate()
        with open(self.path, 'wb') as f:
            dill.dump(self, f)
        print(f"Model saved to {self.path}")


    def run(self):
        print('\n#########################################################################################################################################\n')
        self.train()
        self.evaluate()
        dump(self.path, self)


models = []

In [ ]:
@dataclasses.dataclass
class diab_cf():
    wrangler: any
    learner: any
    param_grid: any
    scorer: any
    target: str = 'GEO_CC'
    n_splits: int = 5
    refit_time: int = 120
    n_repeats: int = 5
    path: str = None


    def __post_init__(self):
        self.now = pd.Timestamp.now()
        self.path = mkdir(self.path if self.path is not None else root/f'models_cf/{self.now}.pkl')

        # make modeling & holdout sets
        X = copy(cf)
        y = X.pop(self.target)
        # Drop rows with missing target values before splitting
        valid_indices = y.dropna().index
        X = X.loc[valid_indices]
        y = y.loc[valid_indices]

        self.X_modeling, self.X_holdout, self.y_modeling, self.y_holdout = train_test_split(X, y, test_size=0.20, stratify=y, random_state=seed)

        # make estimator & grid
        self.learner.__sklearn_tags__ = getattr(sk.base, self.learner._estimator_type.capitalize()+'Mixin')().__sklearn_tags__
        self.estimator = Pipeline((('wrangle', self.wrangler), ('learn', self.learner)))
        self.grid = GridSearchCV(self.estimator, self.param_grid, cv=self.n_splits, scoring=self.scorer, refit=False)


    def train(self):
        # run grid search
        self.grid.fit(self.X_modeling, self.y_modeling)

        # retrain best estimator with larger time_budget and cross-validation if FLAML
        kwargs = {
            'time_budget': self.refit_time,
            'retrain_full': True,
            'eval_method': 'cv',
            'n_splits': self.n_splits,
        }
        kwargs = {f'learn__{k}':v for k,v in kwargs.items() if k in self.learner.get_params()}
        self.estimator = self.estimator.set_params(**self.grid.best_params_).fit(self.X_modeling, self.y_modeling, **kwargs)

        # predict
        self.pred_modeling = self.estimator.predict(self.X_modeling)
        self.pred_holdout  = self.estimator.predict(self.X_holdout)


    def evaluate(self):
        # name of best algorithm
        self.algorithm = getattr(self.estimator['learn'], 'best_estimator', str(self.estimator['learn']).split('(')[0])
        disp(self.algorithm)

        # all results from grid search
        self.grid.cv_df = pd.DataFrame(self.grid.cv_results_['params']).assign(cv_score=self.grid.cv_results_['mean_test_score']).sort_values('cv_score', ascending=False).rename_axis(index='grid_id')
        disp(self.grid.cv_df, None)

        # best estimator's scores
        self.scores = {'cv':self.grid.best_score_, 'holdout':self.scorer(self.estimator, self.X_holdout, self.y_holdout)}
        disp(self.scores)

        # reports
        self.classification_report = classification_report(self.y_holdout, self.pred_holdout)
        disp(self.classification_report)
        ConfusionMatrixDisplay.from_estimator(self.estimator, self.X_holdout, self.y_holdout)
        RocCurveDisplay.from_estimator(self.estimator, self.X_holdout, self.y_holdout)
        plt.show()

        # feature importance (permutation)
        imp = permutation_importance(self.estimator, self.X_modeling, self.y_modeling, n_repeats=self.n_repeats, random_state=seed)
        self.importances = pd.DataFrame(imp.importances.T, columns=self.X_modeling.columns)
        mu = self.importances.mean().sort_values(ascending=False)
        sns.stripplot(self.importances[mu[mu>0].index], orient='h')
        plt.xlabel('permutation importance')
        plt.show()


    def run(self):
        print('\n' + '#'*80)
        self.train()
        self.evaluate()
        with open(self.path, 'wb') as f:
            dill.dump(self, f)
        print(f"Model saved to {self.path}")


models_cf = []

In [ ]:
# Confusion Matrix, ROC Curve and Permutation Importance

print("\n" + "="*70)
print("INSTALLING IMBALANCED-LEARN FOR SMOTE...")
print("="*70)
import subprocess
import sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "imbalanced-learn"])
print("✓ imbalanced-learn installed successfully!\n")

# Set random seed
seed = 42
np.random.seed(seed)
root = Path('.')

# LOAD AND PREPARE DATA

print("Loading and preparing data...")
df = pd.read_csv(root / '/content/drive/MyDrive/Data science II/MyDataDiabetes.csv')

# Define feature lists
continuous_features = ['AGE', 'AnnualHouseholdIncome', 'bmi']
categorical_features = [
    'YourGeneralHealth', 'InsuranceStatus', 'PhyActCate', 'TakingSubsc',
    'HeartDisease', 'HadStroke', 'CardioDisease', 'AsthmaDisease', 'AnyCncer',
    'SEX', 'HispanicLatinoOrigin', 'MarStatus', 'EduStatus', 'EmployStatus',
    'DisabilityStatus', 'SmoleCigga','cfips','state'
]

# Prepare FIPS codes
df['cfips_str'] = '48' + df['cfips'].astype(str).str.replace('.0', '', regex=False).str.zfill(3)
df['state_fips'] = df['cfips_str'].str.slice(0, 2)

# Convert diabetes to binary (1 and 2 → 1 and 0)
df['diabetes'] = df['diabetes'].replace({1: 1, 2: 0})

print(f"Data shape: {df.shape}")
print(f"Diabetes distribution:\n{df['diabetes'].value_counts()}")

# CREATE PREPROCESSING PIPELINE WITH SMOTE

print("Creating preprocessing pipeline with SMOTE...")

features_to_use = continuous_features + categorical_features

# First, create the feature transformer (without SMOTE)
feature_transformer = ColumnTransformer(
    transformers=[
        ('continuous', Pipeline([
            ('impute', SimpleImputer(strategy='mean')),
            ('scale', StandardScaler()),
            ('reddim', PCA(n_components=min(3, len(continuous_features)))),
        ]), continuous_features),

        ('nominal', Pipeline([
            ('impute', SimpleImputer(strategy='most_frequent')),
            ('encode', OneHotEncoder(
                sparse_output=False,
                drop='if_binary',
                handle_unknown='infrequent_if_exist'
            )),
        ]), categorical_features),
    ],
    remainder='drop',
    verbose_feature_names_out=False,
)

# Create SMOTE
smote_instance = SMOTE(
    random_state=seed,
    k_neighbors=3,           # ← CHANGED FROM 5
    sampling_strategy=0.7    # ← CHANGED FROM 'minority'
)

n_continuous_features = len(continuous_features)

# DEFINE DIAB CLASS WITH SMOTE

class diab:
    def __init__(self, feature_transformer_obj, smote_obj, learner, learner_name, target='diabetes', n_splits=5):
        self.feature_transformer_obj = feature_transformer_obj
        self.smote_obj = smote_obj
        self.learner = learner
        self.learner_name = learner_name
        self.target = target
        self.n_splits = n_splits

        self.X_modeling = None
        self.X_holdout = None
        self.y_modeling = None
        self.y_holdout = None
        self.best_model = None
        self.results = None

    def run(self):
        # Create the full ImbPipeline here, avoiding nested ImbPipelines
        self.best_model = ImbPipeline([
            ('preprocessing', self.feature_transformer_obj),
            ('smote', self.smote_obj),
            ('classifier', self.learner)
        ])

        # Fit on modeling data (SMOTE will be applied during fit)
        self.best_model.fit(self.X_modeling, self.y_modeling)

        self.results = {'trained': True}
        return self.results

# PREPARE DATA FOR MODELING

print("\nPreparing data for modeling...")

target = 'diabetes'
valid_indices = df[target].dropna().index
df_cleaned = df.loc[valid_indices].copy()

X = df_cleaned[features_to_use]
y = df_cleaned[target]

print(f"Final dataset shape: {X.shape}")

# TRAIN MODELS WITH EVALUATION

models = []

learners = [
    ('HistGradientBoosting', HistGradientBoostingClassifier(
        random_state=seed,
        max_leaf_nodes=255,      # ← CHANGED FROM 31
        max_depth=15,            # ← ADDED
        learning_rate=0.1,       # ← ADDED
        l2_regularization=0.0    # ← ADDED
    )),
    ('AutoML (Balanced Accuracy)', AutoML(time_budget=60, eval_method='cv', n_splits=5,
                                     task='classification', metric='f1', seed=seed, verbose=1))
]

for learner_name, learner in learners:
    print("\n" + "="*70)
    print(f"TRAINING: {learner_name}")
    print("="*70)

    # Split data
    X_modeling, X_holdout, y_modeling, y_holdout = train_test_split(
        X, y, test_size=0.20, stratify=y, random_state=seed
    )

    print(f"Training set: {X_modeling.shape}")
    print(f"Test set: {X_holdout.shape}")
    print(f"Training distribution:\n{pd.Series(y_modeling).value_counts()}")
    print(f"Test distribution:\n{pd.Series(y_holdout).value_counts()}")

    # Create and train model (passing feature_transformer and smote_instance separately)
    obj = diab(feature_transformer, smote_instance, learner, learner_name, target='diabetes', n_splits=5)
    obj.X_modeling = X_modeling
    obj.X_holdout = X_holdout
    obj.y_modeling = y_modeling
    obj.y_holdout = y_holdout

    print(f"\nTraining {learner_name} with SMOTE...")
    print(f"Original training set distribution:\n{pd.Series(y_modeling).value_counts()}")
    obj.run()

    print(f"✓ {learner_name} training complete!")

    # EVALUATION METRIC 1: PREDICTIONS & BASIC METRICS (F1 FOCUSED)

    print("\n" + "-"*70)
    print("EVALUATION METRICS (F1-FOCUSED)")
    print("-"*70)

    y_pred = obj.best_model.predict(X_holdout)
    y_pred_proba = obj.best_model.predict_proba(X_holdout)[:, 1]

    accuracy = accuracy_score(y_holdout, y_pred)
    precision = precision_score(y_holdout, y_pred, zero_division=0)
    recall = recall_score(y_holdout, y_pred, zero_division=0)
    f1 = f1_score(y_holdout, y_pred, zero_division=0)

    # F1 for each class
    f1_0 = f1_score(y_holdout, y_pred, labels=[0], zero_division=0)
    f1_1 = f1_score(y_holdout, y_pred, labels=[1], zero_division=0)

    print(f"\n📊 OVERALL METRICS:")
    print(f"  Accuracy:  {accuracy:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  F1-Score (Macro):  {f1:.4f}")
    print(f"\n📊 F1-SCORE BY CLASS:")
    print(f"  F1 (No Diabetes):   {f1_0:.4f}")
    print(f"  F1 (Has Diabetes):  {f1_1:.4f}")

    # EVALUATION METRIC 2: CLASSIFICATION REPORT

    print("\n" + "-"*70)
    print("CLASSIFICATION REPORT (DETAILED)")
    print("-"*70)
    print(classification_report(y_holdout, y_pred, target_names=['No Diabetes', 'Has Diabetes'], digits=4))

    # EVALUATION METRIC 3: CONFUSION MATRIX

    print("\n" + "-"*70)
    print("CONFUSION MATRIX ANALYSIS")
    print("-"*70)

    cm = confusion_matrix(y_holdout, y_pred)
    print(f"\n{cm}")

    tn, fp, fn, tp = cm.ravel()
    print(f"\n✓ True Negatives:  {tn}")
    print(f"✓ True Positives:  {tp}")
    print(f"✗ False Positives: {fp}")
    print(f"✗ False Negatives: {fn}")

    # Sensitivity (Recall) and Specificity
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

    print(f"\n📈 DERIVED METRICS:")
    print(f"  Sensitivity (True Positive Rate): {sensitivity:.4f}")
    print(f"  Specificity (True Negative Rate): {specificity:.4f}")

    # Visualize Confusion Matrix
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True, ax=ax,
                xticklabels=['No Diabetes', 'Has Diabetes'],
                yticklabels=['No Diabetes', 'Has Diabetes'],
                annot_kws={'size': 14, 'weight': 'bold'})
    ax.set_ylabel('True Label', fontsize=13, fontweight='bold')
    ax.set_xlabel('Predicted Label', fontsize=13, fontweight='bold')
    ax.set_title(f'Confusion Matrix - {learner_name}\n(F1-Optimized Model)', fontsize=15, fontweight='bold', pad=20)

    # Add metrics text box
    textstr = f'Sensitivity: {sensitivity:.4f}\nSpecificity: {specificity:.4f}\nF1 (Diabetes): {f1_1:.4f}'
    ax.text(0.98, 0.02, textstr, transform=ax.transAxes, fontsize=11,
           verticalalignment='bottom', horizontalalignment='right',
           bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

    plt.tight_layout()
    plt.show()

    # EVALUATION METRIC 4: ROC CURVE

    print("\n" + "-"*70)
    print("ROC CURVE ANALYSIS (FOR DIABETES POSITIVE CLASS)")
    print("-"*70)

    fpr, tpr, thresholds = roc_curve(y_holdout, y_pred_proba)
    roc_auc = auc(fpr, tpr)

    print(f"ROC AUC Score: {roc_auc:.4f}")

    # Find optimal threshold that maximizes F1
    f1_scores = [f1_score(y_holdout, (y_pred_proba >= t).astype(int)) for t in thresholds]
    optimal_idx = np.argmax(f1_scores)
    optimal_threshold = thresholds[optimal_idx]
    print(f"\n🎯 Optimal Threshold (for F1): {optimal_threshold:.4f}")
    print(f"🎯 Default Threshold: 0.5000")

    # ===== THRESHOLD ANALYSIS - TEST MULTIPLE THRESHOLDS =====
    print("\n" + "-"*70)
    print("THRESHOLD SENSITIVITY ANALYSIS")
    print("-"*70)

    threshold_results = []
    test_thresholds = [0.5, 0.4, 0.3, 0.25, 0.2, optimal_threshold]

    for test_thresh in sorted(set(test_thresholds)):
        y_pred_test = (y_pred_proba >= test_thresh).astype(int)
        sens_test = recall_score(y_holdout, y_pred_test)
        spec_test = recall_score(y_holdout, y_pred_test, pos_label=0)
        f1_test = f1_score(y_holdout, y_pred_test)

        cm_test = confusion_matrix(y_holdout, y_pred_test)
        tn_t, fp_t, fn_t, tp_t = cm_test.ravel()

        threshold_results.append({
            'Threshold': f'{test_thresh:.4f}',
            'TP': tp_t,
            'TN': tn_t,
            'FP': fp_t,
            'FN': fn_t,
            'Sensitivity': f'{sens_test:.4f}',
            'Specificity': f'{spec_test:.4f}',
            'F1-Score': f'{f1_test:.4f}'
        })

        print(f"\n📊 Threshold = {test_thresh:.4f}:")
        print(f"   TP: {tp_t} | TN: {tn_t} | FP: {fp_t} | FN: {fn_t}")
        print(f"   Sensitivity: {sens_test:.4f} | Specificity: {spec_test:.4f} | F1: {f1_test:.4f}")

    # Display threshold results table
    threshold_df = pd.DataFrame(threshold_results)
    print("\n" + "-"*70)
    print("THRESHOLD COMPARISON TABLE:")
    print(threshold_df.to_string(index=False))

    # ===== APPLY OPTIMAL THRESHOLD =====
    y_pred_optimized = (y_pred_proba >= optimal_threshold).astype(int)

    # Recalculate metrics with optimized threshold
    sensitivity_opt = recall_score(y_holdout, y_pred_optimized)
    specificity_opt = recall_score(y_holdout, y_pred_optimized, pos_label=0)
    f1_opt = f1_score(y_holdout, y_pred_optimized)

    print(f"\n📊 METRICS WITH OPTIMIZED THRESHOLD ({optimal_threshold:.4f}):")
    print(f"  Sensitivity: {sensitivity_opt:.4f}")
    print(f"  Specificity: {specificity_opt:.4f}")
    print(f"  F1-Score: {f1_opt:.4f}")

    # Update confusion matrix with optimized predictions
    cm_optimized = confusion_matrix(y_holdout, y_pred_optimized)
    tn_opt, fp_opt, fn_opt, tp_opt = cm_optimized.ravel()
    print(f"\n✅ OPTIMIZED CONFUSION MATRIX (Threshold = {optimal_threshold:.4f}):")
    print(f"  True Negatives:  {tn_opt}")
    print(f"  True Positives:  {tp_opt}")
    print(f"  False Positives: {fp_opt}")
    print(f"  False Negatives: {fn_opt}")

    # Visualize OPTIMIZED Confusion Matrix
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(cm_optimized, annot=True, fmt='d', cmap='RdYlGn', cbar=True, ax=ax,
                xticklabels=['No Diabetes', 'Has Diabetes'],
                yticklabels=['No Diabetes', 'Has Diabetes'],
                annot_kws={'size': 14, 'weight': 'bold'})
    ax.set_ylabel('True Label', fontsize=13, fontweight='bold')
    ax.set_xlabel('Predicted Label', fontsize=13, fontweight='bold')
    ax.set_title(f'Confusion Matrix - {learner_name}\n(OPTIMIZED THRESHOLD = {optimal_threshold:.4f})',
                 fontsize=15, fontweight='bold', pad=20)

    # Add metrics text box
    textstr_opt = f'Sensitivity: {sensitivity_opt:.4f}\nSpecificity: {specificity_opt:.4f}\nF1 (Diabetes): {f1_opt:.4f}'
    ax.text(0.98, 0.02, textstr_opt, transform=ax.transAxes, fontsize=11,
           verticalalignment='bottom', horizontalalignment='right',
           bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.8))

    plt.tight_layout()
    plt.show()

    # Visualize ROC Curve
    fig, ax = plt.subplots(figsize=(11, 9))

    ax.plot(fpr, tpr, color='darkorange', lw=3, label=f'ROC Curve (AUC = {roc_auc:.4f})')
    ax.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier (AUC = 0.5000)')
    ax.scatter([fpr[optimal_idx]], [tpr[optimal_idx]], color='red', s=200, marker='o',
              label=f'Optimal Threshold = {optimal_threshold:.4f}', zorder=5)

    ax.set_xlim([-0.02, 1.02])
    ax.set_ylim([-0.02, 1.02])
    ax.set_xlabel('False Positive Rate (1 - Specificity)', fontsize=12, fontweight='bold')
    ax.set_ylabel('True Positive Rate (Sensitivity)', fontsize=12, fontweight='bold')
    ax.set_title(f'ROC Curve - {learner_name}\n(F1-Optimized Model)', fontsize=14, fontweight='bold', pad=20)
    ax.legend(loc="lower right", fontsize=11)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    # EVALUATION METRIC 5: PERMUTATION IMPORTANCE

    print("\n" + "-"*70)
    print("PERMUTATION IMPORTANCE (TOP 15 FEATURES)")
    print("-"*70)

    try:
        perm_importance = permutation_importance(
            obj.best_model, X_holdout, y_holdout,
            n_repeats=15, random_state=seed, n_jobs=-1
        )

        # Create feature importance dataframe
        feature_importance_df = pd.DataFrame({
            'Feature': features_to_use,
            'Importance': perm_importance.importances_mean,
            'Std': perm_importance.importances_std
        }).sort_values('Importance', ascending=False)

        print("\nTop 15 Most Important Features:")
        print(feature_importance_df.head(15).to_string(index=False))

        # Visualize Permutation Importance
        fig, ax = plt.subplots(figsize=(13, 9))
        top_features = feature_importance_df.head(15)

        bars = ax.barh(range(len(top_features)), top_features['Importance'].values,
                       xerr=top_features['Std'].values, capsize=5, color='steelblue', alpha=0.8)
        ax.set_yticks(range(len(top_features)))
        ax.set_yticklabels(top_features['Feature'].values, fontsize=11)
        ax.set_xlabel('Permutation Importance', fontsize=12, fontweight='bold')
        ax.set_title(f'Top 15 Feature Importance - {learner_name}\n(Based on F1 Score Impact)',
                    fontsize=14, fontweight='bold', pad=20)
        ax.grid(True, alpha=0.3, axis='x')

        # Color gradient
        colors_list = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(bars)))
        for bar, color in zip(bars, colors_list):
            bar.set_color(color)

        plt.tight_layout()
        plt.show()

    except Exception as e:
        print(f"Could not compute permutation importance: {e}")

    # Store model
    obj.y_pred = y_pred
    obj.y_pred_proba = y_pred_proba
    obj.roc_auc = roc_auc
    obj.f1_score = f1
    obj.f1_diabetes = f1_1
    obj.optimal_threshold = optimal_threshold
    obj.sensitivity_opt = sensitivity_opt
    obj.specificity_opt = specificity_opt
    obj.f1_opt = f1_opt
    models.append(obj)

# MODEL COMPARISON SUMMARY (F1-FOCUSED)

print("\n" + "="*70)
print("MODEL COMPARISON SUMMARY (F1-OPTIMIZED)")
print("="*70)

comparison_data = []
for i, obj in enumerate(models):
    learner_name = obj.learner_name
    print(f"\n{i+1}. {learner_name}")
    print(f"   DEFAULT THRESHOLD (0.5):")
    print(f"     F1 Score (Overall):      {obj.f1_score:.4f}")
    print(f"     F1 Score (Diabetes):     {obj.f1_diabetes:.4f}")
    print(f"   OPTIMIZED THRESHOLD ({obj.optimal_threshold:.4f}):")
    print(f"     Sensitivity:             {obj.sensitivity_opt:.4f}")
    print(f"     Specificity:             {obj.specificity_opt:.4f}")
    print(f"     F1-Score:                {obj.f1_opt:.4f}")
    print(f"   ROC AUC Score:           {obj.roc_auc:.4f}")

    comparison_data.append({
        'Model': learner_name,
        'Default_F1': f'{obj.f1_score:.4f}',
        'Optimized_Sensitivity': f'{obj.sensitivity_opt:.4f}',
        'Optimized_F1': f'{obj.f1_opt:.4f}',
        'ROC_AUC': f'{obj.roc_auc:.4f}'
    })

comparison_df = pd.DataFrame(comparison_data)
print("\n" + "-"*70)
print("Comparison Table:")
print(comparison_df.to_string(index=False))

print("\n" + "="*70)
print("✓ COMPLETE F1-OPTIMIZED EVALUATION WITH SMOTE FINISHED!")
print("="*70)
print("\nKEY INSIGHTS:")
print("  • Models optimized with SMOTE for balanced class distribution")
print("  • SMOTE: Synthetic Minority Over-Sampling Technique")
print("  • Creates synthetic samples of minority class (Diabetes=1)")
print("  • Improves recall for minority class (Diabetes positive)")
print("  • Better F1 scores for imbalanced datasets")
print("  • Reduces bias towards majority class")
print("  • THRESHOLD OPTIMIZATION is critical for imbalanced problems!")
print("  • Default 0.5 threshold is often too conservative")
print("  • Optimal threshold may be 0.2-0.4 for medical problems")
print("  • Always check multiple thresholds before choosing final model")

In [ ]:
# F1 SCORE ANALYSIS - INDIVIDUAL SUBGROUPS

print("\n" + "="*70)
print("F1 SCORE ANALYSIS - DIABETES POSITIVE CLASS BY INDIVIDUAL SUBGROUPS")
print("="*70)

# Create a copy of holdout data with predictions
results_df = X_holdout.copy()
results_df['y_true'] = y_holdout.values
results_df['y_pred'] = y_pred  # Use predictions from the best performing model

print(f"\nDataset shape: {results_df.shape}")
print(f"Features for subgroup analysis: {categorical_features}")

# FUNCTION: Calculate F1 Score for Each Individual Subgroup

def calculate_f1_by_subgroup(df, feature_col, y_true_col, y_pred_col):
    """
    Calculate F1 score for positive class (diabetes=1) for each unique value in a feature
    """
    f1_scores = []

    unique_values = sorted(df[feature_col].unique())

    for value in unique_values:
        mask = df[feature_col] == value
        subset_y_true = df.loc[mask, y_true_col]
        subset_y_pred = df.loc[mask, y_pred_col]

        if len(subset_y_true) > 0:
            # Calculate F1 score for positive class (diabetes=1)
            f1 = f1_score(subset_y_true, subset_y_pred, pos_label=1, zero_division=0)

            # Calculate additional metrics
            tp = ((subset_y_pred == 1) & (subset_y_true == 1)).sum()
            fp = ((subset_y_pred == 1) & (subset_y_true == 0)).sum()
            fn = ((subset_y_pred == 0) & (subset_y_true == 1)).sum()
            tn = ((subset_y_pred == 0) & (subset_y_true == 0)).sum()

            precision = tp / (tp + fp) if (tp + fp) > 0 else 0
            recall = tp / (tp + fn) if (tp + fn) > 0 else 0

            f1_scores.append({
                'Feature': feature_col,
                'Category': str(value),
                'F1_Score': f1,
                'Precision': precision,
                'Recall': recall,
                'Sample_Size': len(subset_y_true),
                'Positive_Cases': (subset_y_true == 1).sum(),
                'Negative_Cases': (subset_y_true == 0).sum(),
                'TP': tp,
                'FP': fp,
                'FN': fn,
                'TN': tn
            })

    return f1_scores

# CALCULATE F1 SCORES FOR ALL CATEGORICAL FEATURES
all_f1_results = []

print("\nCalculating F1 scores for EACH category in each feature...\n")

for feature in categorical_features:
    if feature in results_df.columns:
        print(f"Processing feature: {feature}")
        feature_results = calculate_f1_by_subgroup(
            results_df, feature, 'y_true', 'y_pred'
        )
        all_f1_results.extend(feature_results)

        # Print individual results for this feature
        print(f"  {'-'*65}")
        for result in feature_results:
            print(f"    Category: {result['Category']:20s} | F1: {result['F1_Score']:.4f} | Precision: {result['Precision']:.4f} | Recall: {result['Recall']:.4f} | n={result['Sample_Size']}")
        print()
    else:
        print(f"  ⚠ Feature {feature} not found in data\n")

# Convert to DataFrame
f1_results_df = pd.DataFrame(all_f1_results)

print(f"✓ F1 scores calculated for {len(f1_results_df)} individual categories")

# DISPLAY DETAILED TABLE FOR EACH CATEGORY

print("\n" + "="*70)
print("DETAILED F1 SCORES FOR EACH INDIVIDUAL CATEGORY")
print("="*70)

for feature in categorical_features:
    if feature in results_df.columns:
        feature_data = f1_results_df[f1_results_df['Feature'] == feature].sort_values('F1_Score', ascending=False)

        print(f"\n{'-'*70}")
        print(f"FEATURE: {feature}")
        print(f"{'-'*70}")
        print(f"{'Category':<30} {'F1 Score':>10} {'Precision':>10} {'Recall':>10} {'Sample Size':>12}")
        print(f"{'-'*70}")

        for idx, row in feature_data.iterrows():
            print(f"{str(row['Category']):<30} {row['F1_Score']:>10.4f} {row['Precision']:>10.4f} {row['Recall']:>10.4f} {int(row['Sample_Size']):>12}")

        mean_f1 = feature_data['F1_Score'].mean()
        print(f"{'-'*70}")
        print(f"{'MEAN (Feature Average)':<30} {mean_f1:>10.4f}")
        print()


# VISUALIZATION 1: F1 SCORES FOR EACH INDIVIDUAL CATEGORY

print("\n" + "-"*70)
print("Creating Visualization 2: F1 Score by Individual Category (Detailed)")
print("-"*70)

# Sort by feature then by F1 score
f1_results_sorted = f1_results_df.sort_values(['Feature', 'F1_Score'], ascending=[True, False])

# Create a larger figure for all subgroups
fig, ax = plt.subplots(figsize=(14, max(12, len(f1_results_sorted) * 0.18)))

# Create labels combining feature and category
labels = [f"{row['Feature']}: {row['Category']}" for idx, row in f1_results_sorted.iterrows()]

# Create horizontal bar chart
bars = ax.barh(range(len(f1_results_sorted)), f1_results_sorted['F1_Score'].values, color='steelblue', alpha=0.8)

# Color gradient based on F1 score
colors_list = plt.cm.RdYlGn(f1_results_sorted['F1_Score'].values)
for bar, color in zip(bars, colors_list):
    bar.set_color(color)

ax.set_yticks(range(len(f1_results_sorted)))
ax.set_yticklabels(labels, fontsize=9)
ax.set_xlabel('F1 Score (Diabetes Positive Class)', fontsize=12, fontweight='bold')
ax.set_title('F1 Score for Diabetes Positive Class - By Individual Category\n(Each unique value within each feature)',
            fontsize=13, fontweight='bold', pad=20)
ax.set_xlim([0, 1.0])
ax.grid(True, alpha=0.3, axis='x')

# Add value labels on bars with sample size
for i, (idx, row) in enumerate(f1_results_sorted.iterrows()):
    ax.text(row['F1_Score'] + 0.02, i, f"F1={row['F1_Score']:.3f} (n={int(row['Sample_Size'])})",
           va='center', fontsize=8)

plt.tight_layout()
plt.show()

# SUMMARY STATISTICS

print("\n" + "="*70)
print("OVERALL SUMMARY STATISTICS")
print("="*70)

print(f"\nOverall F1 Score (all samples combined): {f1_score(results_df['y_true'], results_df['y_pred'], pos_label=1):.4f}")
print(f"\nF1 Scores across ALL individual categories:")
print(f"  Mean:   {f1_results_df['F1_Score'].mean():.4f}")
print(f"  Median: {f1_results_df['F1_Score'].median():.4f}")
print(f"  Std:    {f1_results_df['F1_Score'].std():.4f}")
print(f"  Min:    {f1_results_df['F1_Score'].min():.4f}")
print(f"  Max:    {f1_results_df['F1_Score'].max():.4f}")

# Categories with lowest and highest F1 scores
print(f"\n🔴 CATEGORIES WITH LOWEST F1 SCORES (need improvement):")
lowest_10 = f1_results_df.nsmallest(10, 'F1_Score')[['Feature', 'Category', 'F1_Score', 'Sample_Size']]
for idx, row in lowest_10.iterrows():
    print(f"   {row['Feature']:25s} | {str(row['Category']):20s} | F1={row['F1_Score']:.4f} | n={int(row['Sample_Size'])}")

print(f"\n🟢 CATEGORIES WITH HIGHEST F1 SCORES (performing well):")
highest_10 = f1_results_df.nlargest(10, 'F1_Score')[['Feature', 'Category', 'F1_Score', 'Sample_Size']]
for idx, row in highest_10.iterrows():
    print(f"   {row['Feature']:25s} | {str(row['Category']):20s} | F1={row['F1_Score']:.4f} | n={int(row['Sample_Size'])}")

# EXPORT RESULTS

print("\n" + "="*70)
print("EXPORTING RESULTS")
print("="*70)

f1_results_df.to_csv('f1_scores_by_individual_category.csv', index=False)
print("✓ Saved: f1_scores_by_individual_category.csv")

print("\n" + "="*70)
print("✓ F1 SCORE ANALYSIS COMPLETE!")
print("="*70)
print("\nYou now have:")
print("  ✓ F1 score for EACH individual category within each feature")
print("  ✓ Precision, Recall, and sample size for each category")
print("  ✓ Detailed visualizations and tables")
print("  ✓ CSV exports for further analysis")

In [ ]:
%pip install geopandas shapely
%pip install geodatasets

In [ ]:
# Geographic(Spatial) Analysis - Diabetes Rate
# Set root path
root = Path('.')

# STEP 1: Load Data

print("Loading data...")
df = pd.read_csv(root / '/content/drive/MyDrive/Data science II/MyDataDiabetes.csv')
cf = pd.read_csv(root / '/content/drive/MyDrive/Data science II/DP05Data1.csv')

print(f"Diabetes data shape: {df.shape}")
print(f"Census data shape: {cf.shape}")
print(f"Census columns: {cf.columns.tolist()[:5]}... (showing first 5)")

# STEP 2: Merge Diabetes Data with Census Data (First 3 Columns)

print("\n" + "="*70)
print("STEP 2: MERGE Diabetes Data with Census Data")
print("="*70)

# Keep only first 3 columns from Census data
cf_subset = cf.iloc[:, :3].copy()
cf_subset.columns = ['geo_id', 'geo_code', 'county_name']

print(f"Census columns selected: {cf_subset.columns.tolist()}")
print(f"Census data shape: {cf_subset.shape}")

# Extract FIPS code from geo_id (format: 0500000US48001)
cf_subset['cfips_str'] = cf_subset['geo_id'].str.extract(r'(48\d{3})')[0]

# Prepare diabetes data FIPS codes
df['cfips_str'] = '48' + df['cfips'].astype(str).str.replace('.0', '', regex=False).str.zfill(3)

print(f"\nSample FIPS from Census: {cf_subset['cfips_str'].head(5).tolist()}")
print(f"Sample FIPS from Diabetes: {df['cfips_str'].head(5).tolist()}")

# MERGE the two datasets
df_merged = df.merge(cf_subset[['cfips_str', 'county_name']], on='cfips_str', how='left')

print(f"\n✓ Merge complete!")
print(f"Merged data shape: {df_merged.shape}")
print(f"Merged data columns: {df_merged.columns.tolist()}")

# Check for missing values
missing_counties = df_merged[df_merged['county_name'].isna()]['cfips_str'].nunique()
print(f"Rows with missing county names: {missing_counties}")

# STEP 3: Prepare Merged Data with MICE Forest for Missing Values

print("\n" + "="*70)
print("STEP 3: PREPARE Merged Data (Handle Missing Values with MICE Forest)")
print("="*70)

print(f"\nFirst few rows of merged data:")
print(df_merged[['diabetes', 'cfips_str', 'county_name']].head(10))

# Check missing values BEFORE imputation
print(f"\n--- MISSING VALUES BEFORE IMPUTATION ---")
missing_before = df_merged.isnull().sum()
print(missing_before[missing_before > 0])
print(f"Missing county names: {df_merged['county_name'].isna().sum()}")

# Check data types and unique values
print(f"\nData types:")
print(df_merged[['diabetes', 'cfips_str', 'county_name']].dtypes)

# CHECK: What are the unique values in the diabetes column?
print(f"\nUnique values in 'diabetes' column:")
print(f"Count: {df_merged['diabetes'].nunique()}")
print(f"Values: {sorted(df_merged['diabetes'].unique())}")
print(f"Value distribution:")
print(df_merged['diabetes'].value_counts().sort_index())

# FIX: Convert diabetes to binary (1 and 2 to 1 and 0)
# The original data has 1 (has diabetes) and 2 (no diabetes).
# We want 1 (has diabetes) and 0 (no diabetes).
if set(df_merged['diabetes'].dropna().unique()).issubset({1, 2}): # Only apply if original values are 1 or 2
    print("\n⚠ Converting diabetes: 1→1 (has diabetes), 2→0 (no diabetes)")
    df_merged['diabetes'] = df_merged['diabetes'].replace({1: 1, 2: 0})
    print(f"After conversion - Value distribution:")
    print(df_merged['diabetes'].value_counts().sort_index())
else:
    print("\nℹ 'diabetes' column not in 1/2 format, skipping direct 1->1, 2->0 conversion.")
    # If it's already 0/1 or some other format, ensure it's binary.
    # This line should convert any non-zero value to 1 and zero to 0.
    # It's a fallback or for safety if other values were present.
    df_merged['diabetes'] = (df_merged['diabetes'] > 0).astype(int)
    print(f"After fallback conversion - Value distribution:")
    print(df_merged['diabetes'].value_counts().sort_index())


# APPLY MICE FOREST FOR MISSING VALUE IMPUTATION

print("\n" + "-"*70)
print("APPLYING MICE FOREST IMPUTATION FOR MISSING VALUES")
print("-"*70)

try:
    from sklearn.experimental import enable_iterative_imputer
    from sklearn.impute import IterativeImputer
    from sklearn.ensemble import RandomForestRegressor

    print("Installing and importing MICE Forest...")

    # Create a copy for imputation
    df_impute = df_merged.copy()

    # For categorical column (county_name), we'll use a different strategy
    # First, handle county_name by mapping cfips_str to county_name
    print("\nHandling county_name missing values...")

    # Create a mapping of cfips_str to county_name from non-missing values
    cfips_to_county = df_impute[df_impute['county_name'].notna()].drop_duplicates('cfips_str')[['cfips_str', 'county_name']].set_index('cfips_str')['county_name'].to_dict()

    print(f"County mappings found: {len(cfips_to_county)}")

    # Fill missing county_name using cfips_str mapping
    before_fill = df_impute['county_name'].isna().sum()
    df_impute['county_name'] = df_impute.apply(
        lambda row: cfips_to_county.get(row['cfips_str'], row['county_name']),
        axis=1
    )
    after_fill = df_impute['county_name'].isna().sum()

    print(f"County names filled: {before_fill - after_fill} rows")
    print(f"Remaining missing county names: {after_fill}")

    # For remaining missing values in other columns, use MICE Forest
    numeric_cols = df_impute.select_dtypes(include=[np.number]).columns.tolist()

    if len(numeric_cols) > 0:
        print(f"\nApplying MICE Forest imputation on {len(numeric_cols)} numeric columns...")

        # Use IterativeImputer with RandomForestRegressor as estimator
        imputer = IterativeImputer(
            estimator=RandomForestRegressor(n_estimators=10, random_state=42),
            max_iter=10,
            random_state=42,
            verbose=0
        )

        df_impute[numeric_cols] = imputer.fit_transform(df_impute[numeric_cols])
        print("✓ MICE Forest imputation complete!")

    # Fill any remaining missing values with forward fill or "Unknown"
    df_impute['county_name'] = df_impute['county_name'].fillna('Unknown County')

    df_merged = df_impute

    # Check missing values AFTER imputation
    print(f"\n--- MISSING VALUES AFTER IMPUTATION ---")
    missing_after = df_merged.isnull().sum()
    print(missing_after[missing_after > 0])
    if (missing_after == 0).all():
        print("✓ All missing values imputed successfully!")

except ImportError:
    print("⚠ MICE Forest not available, using simpler imputation...")

    # Fallback: Simple imputation
    # For county_name
    cfips_to_county = df_merged[df_merged['county_name'].notna()].drop_duplicates('cfips_str')[['cfips_str', 'county_name']].set_index('cfips_str')['county_name'].to_dict()
    df_merged['county_name'] = df_merged.apply(
        lambda row: cfips_to_county.get(row['cfips_str'], 'Unknown County'),
        axis=1
    )

    # For numeric columns, use mean imputation
    numeric_cols = df_merged.select_dtypes(include=[np.number]).columns.tolist()
    for col in numeric_cols:
        if df_merged[col].isna().sum() > 0:
            df_merged[col] = df_merged[col].fillna(df_merged[col].mean())

    print("✓ Simple imputation complete!")

print(f"✓ Merged data prepared!")

# STEP 4: Calculate Merged Data Statistics by County

print("\n" + "="*70)
print("STEP 4: CALCULATE Merged Data Statistics by County")
print("="*70)

# Group by county and calculate statistics
merged_by_county = df_merged.groupby('cfips_str').agg({
    'diabetes': ['mean', 'count', 'sum'],
    'county_name': 'first'
}).reset_index()

merged_by_county.columns = ['cfips_str', 'diabetes_rate', 'total_patients', 'diabetes_cases', 'county_name']

# Convert prevalence to percentage
merged_by_county['diabetes_rate'] = merged_by_county['diabetes_rate'] * 100

# Reorder columns
merged_by_county = merged_by_county[['cfips_str', 'county_name', 'diabetes_rate', 'total_patients', 'diabetes_cases']]

print(f"Statistics calculated for {len(merged_by_county)} counties")
print(f"\nSample statistics:")
print(merged_by_county.head(15))

# Overall statistics
print("\n" + "="*70)
print("OVERALL DIABETES STATISTICS")
print("="*70)
print(f"Total patients: {merged_by_county['total_patients'].sum():,}")
print(f"Total diabetes cases: {merged_by_county['diabetes_cases'].sum():,}")
print(f"Overall diabetes rate: {(merged_by_county['diabetes_cases'].sum() / merged_by_county['total_patients'].sum() * 100):.2f}%")
print(f"\nDiabetes Rate by County (Statistics):")
print(merged_by_county['diabetes_rate'].describe())

# Top 15 counties with the highest diabetes prevalence
print("\n" + "-"*70)
print("TOP 15 COUNTIES WITH HIGHEST DIABETES RATE:")
print("-"*70)
top_15 = merged_by_county.nlargest(15, 'diabetes_rate')
for idx, (i, row) in enumerate(top_15.iterrows(), 1):
    print(f"{idx:2d}. {row['county_name']:30s} : {row['diabetes_rate']:6.2f}% (n={int(row['total_patients']):5d})")

# Top 15 counties with lowest diabetes prevalence
print("\n" + "-"*70)
print("TOP 15 COUNTIES WITH LOWEST DIABETES RATE:")
print("-"*70)
bottom_15 = merged_by_county[merged_by_county['diabetes_rate'] > 0].nsmallest(15, 'diabetes_rate')
for idx, (i, row) in enumerate(bottom_15.iterrows(), 1):
    print(f"{idx:2d}. {row['county_name']:30s} : {row['diabetes_rate']:6.2f}% (n={int(row['total_patients']):5d})")

# STEP 5: Download Texas Shapefile

print("\n" + "="*70)
print("STEP 5: Download Texas Shapefile")
print("="*70)

try:
    print("Downloading Texas county boundaries...")
    url = "https://www2.census.gov/geo/tiger/GENZ2020/shp/cb_2020_us_county_500k.zip"

    response = requests.get(url, timeout=30)
    response.raise_for_status()

    with zipfile.ZipFile(io.BytesIO(response.content)) as zip_ref:
        zip_ref.extractall('temp_shapefiles')

    # Read the shapefile
    texas_gdf = gpd.read_file('temp_shapefiles/cb_2020_us_county_500k.shp')

    # Filter for Texas (FIPS 48)
    texas_gdf = texas_gdf[texas_gdf['STATEFP'] == '48'].copy()

    # Create FIPS code column
    texas_gdf['cfips_str'] = texas_gdf['STATEFP'] + texas_gdf['COUNTYFP']

    print(f"✓ Loaded Texas data! {len(texas_gdf)} counties")

    # STEP 6: Merge with Geographic Data and Create Visualization

    print("\n" + "="*70)
    print("STEP 6: CREATE VISUALIZATIONS FOR MERGED DATA")
    print("="*70)

    texas_merged = texas_gdf.merge(merged_by_county, on='cfips_str', how='left')

    print(f"Geographic merged data shape: {texas_merged.shape}")
    print(f"Counties with diabetes data: {texas_merged['diabetes_rate'].notna().sum()}")

    # Visualization 1: Diabetes Prevalence Map
    print("\nCreating Visualization 1: Diabetes Rate Map...")
    fig, ax = plt.subplots(figsize=(18, 14))
    texas_merged.plot(
        column='diabetes_rate',
        cmap='YlOrRd',
        legend=True,
        ax=ax,
        edgecolor='k',
        linewidth=0.3,
        vmin=merged_by_county['diabetes_rate'].quantile(0.05),
        vmax=merged_by_county['diabetes_rate'].quantile(0.95),
        legend_kwds={
            'label': 'Diabetes Rate (%)',
            'orientation': 'vertical',
            'shrink': 0.8
        }
    )
    ax.set_title('Diabetes Rate by Texas County (Merged Data)', fontsize=18, fontweight='bold', pad=20)
    ax.set_xlabel('Longitude', fontsize=12)
    ax.set_ylabel('Latitude', fontsize=12)
    ax.axis('off')
    plt.tight_layout()
    plt.show()

    # Visualization 2: Sample Size Map
    print("Creating Visualization 2: Sample Size Map...")
    fig, ax = plt.subplots(figsize=(18, 14))
    texas_merged.plot(
        column='total_patients',
        cmap='Blues',
        legend=True,
        ax=ax,
        edgecolor='k',
        linewidth=0.3,
        legend_kwds={
            'label': 'Number of Patients',
            'orientation': 'vertical',
            'shrink': 0.8
        }
    )
    ax.set_title('Sample Size by Texas County (Merged Data)', fontsize=18, fontweight='bold', pad=20)
    ax.set_xlabel('Longitude', fontsize=12)
    ax.set_ylabel('Latitude', fontsize=12)
    ax.axis('off')
    plt.tight_layout()
    plt.show()


except Exception as e:
    print(f"\nError: {e}")
    import traceback
    traceback.print_exc()

    print("\nFallback: Creating visualizations without geographic data...")

In [ ]:
# Visualization of Top and Bottom 15 Counties

# Visualization: Top 15 Counties Bar Chart with Description
print("Creating Visualization 3: Top 15 Counties Bar Chart...")
fig, ax = plt.subplots(figsize=(16, 10))
top_15 = merged_by_county.nlargest(15, 'diabetes_rate')
bars = ax.barh(range(len(top_15)), top_15['diabetes_rate'].values, color='steelblue')

colors_list = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(bars)))
for bar, color in zip(bars, colors_list):
    bar.set_color(color)

ax.set_yticks(range(len(top_15)))
ax.set_yticklabels(top_15['county_name'].values, fontsize=11)
ax.set_xlabel('Diabetes Rate (%)', fontsize=13, fontweight='bold')
ax.set_title('Top 15 Texas Counties with Highest Diabetes Rate (Merged Data)',
             fontsize=15, fontweight='bold', pad=20)
ax.set_xlim(0, 105)
ax.grid(True, alpha=0.3, axis='x')

# Add value labels on bars
for i, (idx, row) in enumerate(top_15.iterrows()):
    ax.text(row['diabetes_rate'] + 1, i, f"{row['diabetes_rate']:.1f}%",
           va='center', fontsize=10, fontweight='bold')

# Add comprehensive text box with interpretation
textstr = f'''KEY FINDINGS:\n\n• Caldwell, Burnet, and Burleson Counties lead with ~100% diabetes rate\n• These 15 counties show diabetes rate ranging from {top_15['diabetes_rate'].min():.1f}% to {top_15['diabetes_rate'].max():.1f}%\n• Color gradient (red→green) indicates rate level: darker red = higher rate\n• Total patients in top 15 counties: {int(top_15['total_patients'].sum()):,}\n\nINTERPRETATION:\n✓ High rate indicates most sampled patients in these counties have diabetes\n✓ May reflect: (1) High disease burden, (2) Health-seeking behavior, or (3) Data collection bias\n✓ Anderson County: 100% rate but only n=4 (small sample - less reliable)\n✓ Angelina County: 99.4% rate with n=472 (larger sample - more reliable)\n\nIMPLICATION FOR PUBLIC HEALTH:\n→ Counties with high rate need targeted diabetes prevention & management programs\n→ Consider sample size when interpreting results (larger n = more reliable)\n→ Geographic clustering may indicate socioeconomic or lifestyle factors\n    '''

ax.text(0.98, -0.15, textstr, transform=ax.transAxes, fontsize=10,
           verticalalignment='top', horizontalalignment='right',
           bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8), family='monospace')

plt.tight_layout()
plt.subplots_adjust(bottom=0.35)
plt.show()

# Visualization: Bottom 15 Counties Bar Chart with Description
print("Creating Visualization 3B: Bottom 15 Counties Bar Chart...")
fig, ax = plt.subplots(figsize=(16, 10))

# Get bottom 15 counties with diabetes rate > 0 (to avoid zero-rate outliers)
bottom_15 = merged_by_county[merged_by_county['diabetes_rate'] > 0].nsmallest(15, 'diabetes_rate')

bars = ax.barh(range(len(bottom_15)), bottom_15['diabetes_rate'].values, color='steelblue')

# Color gradient: Green for low rates (healthy), transitions to yellow
colors_list = plt.cm.YlGn(np.linspace(0.3, 0.9, len(bars)))
for bar, color in zip(bars, colors_list):
    bar.set_color(color)

ax.set_yticks(range(len(bottom_15)))
ax.set_yticklabels(bottom_15['county_name'].values, fontsize=11)
ax.set_xlabel('Diabetes Rate (%)', fontsize=13, fontweight='bold')
ax.set_title('Top 15 Texas Counties with Lowest Diabetes Rate (Merged Data)',
             fontsize=15, fontweight='bold', pad=20)
ax.set_xlim(0, max(bottom_15['diabetes_rate'].max() + 5, 30))
ax.grid(True, alpha=0.3, axis='x')

# Add value labels on bars
for i, (idx, row) in enumerate(bottom_15.iterrows()):
    ax.text(row['diabetes_rate'] + 0.5, i, f"{row['diabetes_rate']:.1f}%",
           va='center', fontsize=10, fontweight='bold')

# Add comprehensive text box with interpretation
textstr = f'''KEY FINDINGS:\n\n• {bottom_15.iloc[0]['county_name']}, {bottom_15.iloc[1]['county_name']}, and {bottom_15.iloc[2]['county_name']} have the lowest rates\n• These 15 counties show diabetes rate ranging from {bottom_15['diabetes_rate'].min():.1f}% to {bottom_15['diabetes_rate'].max():.1f}%\n• Color gradient (yellow→green) indicates rate level: lighter green = lower rate\n• Total patients in bottom 15 counties: {int(bottom_15['total_patients'].sum()):,}\n\nINTERPRETATION:\n✓ Low rate indicates fewer sampled patients have diabetes in these counties\n✓ May reflect: (1) Lower disease burden, (2) Healthier lifestyle, (3) Younger population, or (4) Data collection bias\n✓ Counties with n<30 should be interpreted cautiously (small sample = unreliable)\n✓ Sample size matters: larger n = more reliable rate estimates\n\nCONTRAST WITH HIGH-RATE COUNTIES:\n→ High-rate counties (95-100%): Need intensive intervention programs\n→ Low-rate counties (5-20%): Can serve as models for best practices\n→ Geographic variation suggests modifiable risk factors (diet, exercise, income)\n→ Share successful prevention strategies between low and high-rate counties\n    '''

ax.text(0.98, -0.18, textstr, transform=ax.transAxes, fontsize=10,
       verticalalignment='top', horizontalalignment='right',
       bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8), family='monospace')

plt.tight_layout()
plt.subplots_adjust(bottom=0.40)
plt.show()


# Create a combined comparison visualization
print("\nCreating Combined Comparison: Highest vs Lowest Diabetes Rate Counties...")
fig, axes = plt.subplots(1, 2, figsize=(20, 10))

# Top 15 (High Rate)
top_15 = merged_by_county.nlargest(15, 'diabetes_rate')
bars_top = axes[0].barh(range(len(top_15)), top_15['diabetes_rate'].values, color='steelblue')
colors_top = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(bars_top)))
for bar, color in zip(bars_top, colors_top):
    bar.set_color(color)

axes[0].set_yticks(range(len(top_15)))
axes[0].set_yticklabels(top_15['county_name'].values, fontsize=10)
axes[0].set_xlabel('Diabetes Rate (%)', fontsize=12, fontweight='bold')
axes[0].set_title('Top 15: HIGHEST Diabetes Rate', fontsize=14, fontweight='bold', color='darkred')
axes[0].set_xlim(0, 105)
axes[0].grid(True, alpha=0.3, axis='x')

for i, (idx, row) in enumerate(top_15.iterrows()):
    axes[0].text(row['diabetes_rate'] + 1, i, f"{row['diabetes_rate']:.1f}%",
                va='center', fontsize=9, fontweight='bold')

# Bottom 15 (Low Rate)
bottom_15_clean = merged_by_county[merged_by_county['diabetes_rate'] > 0].nsmallest(15, 'diabetes_rate')
bars_bottom = axes[1].barh(range(len(bottom_15_clean)), bottom_15_clean['diabetes_rate'].values, color='steelblue')
colors_bottom = plt.cm.YlGn(np.linspace(0.3, 0.9, len(bars_bottom)))
for bar, color in zip(bars_bottom, colors_bottom):
    bar.set_color(color)

axes[1].set_yticks(range(len(bottom_15_clean)))
axes[1].set_yticklabels(bottom_15_clean['county_name'].values, fontsize=10)
axes[1].set_xlabel('Diabetes Rate (%)', fontsize=12, fontweight='bold')
axes[1].set_title('Top 15: LOWEST Diabetes Rate', fontsize=14, fontweight='bold', color='darkgreen')
axes[1].set_xlim(0, max(bottom_15_clean['diabetes_rate'].max() + 5, 30))
axes[1].grid(True, alpha=0.3, axis='x')

for i, (idx, row) in enumerate(bottom_15_clean.iterrows()):
    axes[1].text(row['diabetes_rate'] + 0.5, i, f"{row['diabetes_rate']:.1f}%",
                va='center', fontsize=9, fontweight='bold')

# Add overall comparison statistics
fig.suptitle('Texas County Diabetes Rate Extremes Comparison', fontsize=16, fontweight='bold', y=0.98)

comparison_text = f'''
STATISTICAL COMPARISON:

Highest 15 Counties:
  • Mean rate: {top_15['diabetes_rate'].mean():.2f}%
  • Range: {top_15['diabetes_rate'].min():.2f}% - {top_15['diabetes_rate'].max():.2f}%
  • Total patients: {int(top_15['total_patients'].sum()):,}
  • Total diabetes cases: {int(top_15['diabetes_cases'].sum()):,}

Lowest 15 Counties:
  • Mean rate: {bottom_15_clean['diabetes_rate'].mean():.2f}%
  • Range: {bottom_15_clean['diabetes_rate'].min():.2f}% - {bottom_15_clean['diabetes_rate'].max():.2f}%
  • Total patients: {int(bottom_15_clean['total_patients'].sum()):,}
  • Total diabetes cases: {int(bottom_15_clean['diabetes_cases'].sum()):,}

Rate Difference:
  • Absolute difference: {(top_15['diabetes_rate'].mean() - bottom_15_clean['diabetes_rate'].mean()):.2f} percentage points
  • High-rate counties have {(top_15['diabetes_rate'].mean() / bottom_15_clean['diabetes_rate'].mean()):.1f}x higher diabetes prevalence
'''

fig.text(0.5, -0.05, comparison_text, ha='center', fontsize=10,
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9),
         family='monospace')

plt.tight_layout()
plt.subplots_adjust(bottom=0.20)
plt.show()


# Statistical summary table
print("\n" + "="*80)
print("STATISTICAL SUMMARY: HIGHEST vs LOWEST DIABETES RATE COUNTIES")
print("="*80)

summary_data = {
    'Metric': [
        'Number of Counties',
        'Mean Diabetes Rate (%)',
        'Min Diabetes Rate (%)',
        'Max Diabetes Rate (%)',
        'Total Patients',
        'Total Diabetes Cases',
        'Average Sample Size per County'
    ],
    'Highest 15 Counties': [
        len(top_15),
        f"{top_15['diabetes_rate'].mean():.2f}",
        f"{top_15['diabetes_rate'].min():.2f}",
        f"{top_15['diabetes_rate'].max():.2f}",
        f"{int(top_15['total_patients'].sum()):,}",
        f"{int(top_15['diabetes_cases'].sum()):,}",
        f"{top_15['total_patients'].mean():.0f}"
    ],
    'Lowest 15 Counties': [
        len(bottom_15_clean),
        f"{bottom_15_clean['diabetes_rate'].mean():.2f}",
        f"{bottom_15_clean['diabetes_rate'].min():.2f}",
        f"{bottom_15_clean['diabetes_rate'].max():.2f}",
        f"{int(bottom_15_clean['total_patients'].sum()):,}",
        f"{int(bottom_15_clean['diabetes_cases'].sum()):,}",
        f"{bottom_15_clean['total_patients'].mean():.0f}"
    ]
}

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))

print("\n" + "="*80)
print("KEY INSIGHTS:")
print("="*80)
print(f"✓ Diabetes rate varies significantly across Texas counties")
print(f"✓ Highest-rate counties have {(top_15['diabetes_rate'].mean() / bottom_15_clean['diabetes_rate'].mean()):.1f}x higher prevalence than lowest")
print(f"✓ This {(top_15['diabetes_rate'].mean() - bottom_15_clean['diabetes_rate'].mean()):.1f} percentage-point gap suggests:")
print(f"  → Geographic/demographic factors influence diabetes prevalence")
print(f"  → Opportunity for targeted public health interventions")
print(f"  → Need to study what makes low-rate counties successful")